In [177]:
from dataclasses import dataclass, field
from typing import List, Dict, Optional
import pandas as pd
import numpy as np
import rasterio
import geopandas as gpd
from datetime import datetime
import matplotlib.pyplot as plt
import sys
sys.path.append('/Users/michaelfoley/Library/CloudStorage/GoogleDrive-mfoley@g.harvard.edu/My Drive/Subnational_Yield_Database/scripts/global/boundaries_processing')
import relationships
import seaborn as sns

In [178]:
from datetime import datetime

current_date = datetime.now().strftime('%m%d%y')
print(current_date)

010926


# Assess what states and districts we have as of the 2001 census
Here we will use the list of towns and villages from the 2001 census (PC01_A_ALT on the census tables website) to get all of the district names

In [179]:
census_df = pd.read_excel('../shapefiles/district_name_files_2001/Alphabetical_List_of_Towns_2001_Table_For_India.xls',
                          header=2, sheet_name='Alphabetical list of Towns')

In [180]:
census_df['State'] = census_df['State'].str.strip()
census_df['District'] = census_df['District'].str.strip()

# Replace substring '&' with 'and' in all columns
census_df = census_df.apply(lambda col: col.str.replace('&', 'and', regex=False) if col.dtype == 'object' else col)

census_df.sort_values(by=['State', 'District'], inplace=True)

# Replace substring in State column specifically
census_df['State'] = census_df['State'].str.replace('Andaman and Nicobar Island', 'Andaman and Nicobar Islands', regex=False)

census_df['State'] = census_df['State'].str.title()
census_df['District'] = census_df['District'].str.title()

census_names = census_df[['State', 'District']].drop_duplicates()

In [181]:
# Fix some issues

# Remove ones with *
census_names = census_names[~census_names['District'].str.contains('*', na=False, regex=False)]

#Fix spelling
census_names.loc[(census_names['State'] == 'Bihar') & (census_names['District'] == 'Kaimur (Bhabua)+C5417'), 
                 'District'] = 'Kaimur (Bhabua)'
census_names.loc[(census_names['State'] == 'Andhra Pradesh') & (census_names['District'] == 'Vi/Sakhapatnam'), 
                 'District'] = 'Visakhapatnam'
census_names.loc[(census_names['State'] == 'Andhra Pradesh') & (census_names['District'] == 'Vi/Zianagaram'), 
                 'District'] = 'Vizianagaram'
census_names.loc[(census_names['State'] == 'Tamil Nadu') & (census_names['District'] == 'Vi/Rudhunagar'), 
                 'District'] = 'Virudhunagar'
census_names.loc[(census_names['State'] == 'Tamil Nadu') & (census_names['District'] == 'Vi/Luppuram'), 
                 'District'] = 'Viluppuram'
census_names.loc[(census_names['State'] == 'Madhya Pradesh') & (census_names['District'] == 'Vi/Disha'), 
                 'District'] = 'Vidisha'
census_names.loc[(census_names['State'] == 'Uttar Pradesh') & (census_names['District'] == 'Sant Ravi/Das Nagar Bhadohi'), 
                 'District'] = 'Sant Ravidas Nagar'

#change Delhi M. Corp. to Central
census_names.loc[(census_names['State'] == 'Delhi') & (census_names['District'] == 'M. Corp.'), 
                 'District'] = 'Central'

# Remove misspelling of Rae Bareli
census_names = census_names[census_names['District'] != 'Rae Bareilly']

#Add Nicobars, Andaman and Nicobar Islands
nicobars = pd.DataFrame({
    'State': ['Andaman And Nicobar Islands'],
    'District': ['Nicobars']
})

census_names = pd.concat([census_names, nicobars], ignore_index=True)

#Add in Upper Siang, Arunachal Pradesh
arunachal_pradesh = pd.DataFrame({
    'State': ['Arunachal Pradesh'],
    'District': ['Upper Siang']
})

census_names = pd.concat([census_names, arunachal_pradesh], ignore_index=True)

# Remove multiple spaces, replacing them with single spaces
census_names['State'] = census_names['State'].str.replace(r'\s+', ' ', regex=True)
census_names['District'] = census_names['District'].str.replace(r'\s+', ' ', regex=True)

#Add alias column and change names for Tamil Nadu
census_names['Alias'] = ''
census_names.loc[(census_names['State'] == 'Uttar Pradesh') & (census_names['District'] == 'Sant Ravidas Nagar'),
                 'Alias'] = 'Bhadohi'

#Drop empty rows
census_names = census_names.dropna(subset=['State', 'District'])

In [182]:
census_names.sort_values(by=['State', 'District'], inplace=True, ignore_index=True)
print(census_names)

                           State                    District Alias
0    Andaman And Nicobar Islands                    Andamans      
1    Andaman And Nicobar Islands                    Nicobars      
2                 Andhra Pradesh                    Adilabad      
3                 Andhra Pradesh                   Anantapur      
4                 Andhra Pradesh                    Chittoor      
..                           ...                         ...   ...
580                  West Bengal                       Nadia      
581                  West Bengal  North Twenty Four Parganas      
582                  West Bengal                    Puruliya      
583                  West Bengal  South Twenty Four Parganas      
584                  West Bengal              Uttar Dinajpur      

[585 rows x 3 columns]


# Load in the shapefile from IndiaStateStories to see where discrepanices lie

In [183]:
india_2001_gdf = gpd.read_file('../shapefiles/India-State-Dist-2001/India_Atlas2001-32.shp')

In [184]:
india_2001_gdf['State'] = india_2001_gdf['State'].str.strip()
india_2001_gdf['District'] = india_2001_gdf['District'].str.strip()


india_2001_gdf['State'] = india_2001_gdf['State'].str.title()
india_2001_gdf['District'] = india_2001_gdf['District'].str.title()

india_2001_gdf['State'] = india_2001_gdf['State'].str.replace(r'\s+', ' ', regex=True)
india_2001_gdf['District'] = india_2001_gdf['District'].str.replace(r'\s+', ' ', regex=True)


In [185]:
india_2001_gdf

,Id,State,District,geometry
0,24,Rajasthan,Bhilwara,"POLYGON ((531189.462 2843344.211, 531831.231 2..."
1,7,Jammu And Kashmir,Leh (Ladakh),"MULTIPOLYGON (((1723180.83 2297750.789, 172256..."
2,14,Punjab,Bathinda,"POLYGON ((469674.59 3302196.61, 467078.326 331..."
3,2,Himachal Pradesh,Kangra,"POLYGON ((648055.425 3521279.552, 646799.258 3..."
4,10,Himachal Pradesh,Sirmaur,"POLYGON ((761194.779 3403911.3, 759507.905 339..."
...,...,...,...,...
589,8,Delhi,South West,"POLYGON ((141277.619 1624160.206, 137507.299 1..."
590,9,Delhi,South,"POLYGON ((164296.415 1564430.399, 163899.539 1..."
591,4,Delhi,East,"POLYGON ((246681.062 1644637.351, 241427.241 1..."
592,5,Delhi,New Delhi,"POLYGON ((197145.731 1602743.104, 199419.922 1..."


In [186]:
unique_to_census = census_names.merge(india_2001_gdf[['State', 'District']],
                                        on=['State', 'District'],
                                        how='left', indicator=True).query('_merge == "left_only"').drop('_merge', axis=1)

unique_to_shapefile = india_2001_gdf.merge(census_names[['State', 'District']],
                                        on=['State', 'District'],
                                        how='left', indicator=True).query('_merge == "left_only"').drop('_merge', axis=1)

In [187]:
print(unique_to_census)

       State     District Alias
147  Gujarat       Rajkot      
337  Manipur  Imphal East      


In [188]:
sorted = unique_to_shapefile.sort_values(by=['State', 'District'])

In [189]:
print(sorted)

     Id             State           District  \
229  23           Gujarat          The Dangs   
582   0           Gujarat               None   
6    12  Himachal Pradesh            Kinnaur   
7     3  Himachal Pradesh    Lahul And Spiti   
123  22       Maharashtra  Mumbai (Suburban)   
550   3           Manipur      Churachandpur   
552   1           Manipur           Senapati   
549   2           Manipur         Tamenglong   
548   8           Manipur             Ukhrul   
583   0           Manipur               None   
540   7           Mizoram          Lawngtlai   
581   0              None               None   

                                              geometry  
229  POLYGON ((386453.174 2319217.303, 386788.066 2...  
582  POLYGON ((99674.404 2629126.769, 100203.572 26...  
6    POLYGON ((817498.755 3458328.416, 816220.022 3...  
7    POLYGON ((767004.761 3516086.651, 767073.57 35...  
123  POLYGON ((289060.117 2122190.744, 288215.559 2...  
550  POLYGON ((2426022.171 283199

In [190]:
#Great - these are all the districts that we seem to have missed in the census data. We can add them in manually.
#Add in manipur
manipur = pd.DataFrame({
    'State': ['Manipur']*4,
    'District': ['Churachandpur', 'Senapati', 'Tamenglong', 'Ukhrul']
})
census_names = pd.concat([census_names, manipur], ignore_index=True)

#Add in Mizoram
mizoram = pd.DataFrame({
    'State': ['Mizoram'],
    'District': ['Lawngtlai']
})
census_names = pd.concat([census_names, mizoram], ignore_index=True)

#Add in Maharashtra district
maharashtra = pd.DataFrame({
    'State': ['Maharashtra'],
    'District': ['Mumbai Suburban']
})
census_names = pd.concat([census_names, maharashtra], ignore_index=True)    

#Add in Gujarat district
gujarat = pd.DataFrame({
    'State': ['Gujarat'],
    'District': ['The Dangs']
})
census_names = pd.concat([census_names, gujarat], ignore_index=True)

#Add in Himachal Pradesh districts
himachal_pradesh = pd.DataFrame({
    'State': ['Himachal Pradesh']*2,
    'District': ['Lahul and Spiti', 'Kinnaur']
})
census_names = pd.concat([census_names, himachal_pradesh], ignore_index=True)

In [191]:
#Check again
unique_to_census = census_names.merge(india_2001_gdf[['State', 'District']],
                                        on=['State', 'District'],
                                        how='left', indicator=True).query('_merge == "left_only"').drop('_merge', axis=1)

unique_to_shapefile = india_2001_gdf.merge(census_names[['State', 'District']],
                                        on=['State', 'District'],
                                        how='left', indicator=True).query('_merge == "left_only"').drop('_merge', axis=1)

In [192]:
print(unique_to_census)

                State         District Alias
147           Gujarat           Rajkot      
337           Manipur      Imphal East      
590       Maharashtra  Mumbai Suburban   NaN
592  Himachal Pradesh  Lahul and Spiti   NaN


In [193]:
print(unique_to_shapefile)

     Id             State           District  \
7     3  Himachal Pradesh    Lahul And Spiti   
123  22       Maharashtra  Mumbai (Suburban)   
581   0              None               None   
582   0           Gujarat               None   
583   0           Manipur               None   

                                              geometry  
7    POLYGON ((767004.761 3516086.651, 767073.57 35...  
123  POLYGON ((289060.117 2122190.744, 288215.559 2...  
581  POLYGON ((713493.923 3185572.512, 713536.569 3...  
582  POLYGON ((99674.404 2629126.769, 100203.572 26...  
583  MULTIPOLYGON (((2339420.997 2850404.254, 23350...  


In [ ]:
census_names.to_csv('census_2001_districts.csv', index=False)